# Generate Test Sentences with TTS Models

In [ ]:
!pip install -U -q kittenml soundfile pandas
!pip install -U -q "transformers>=4.52" accelerate torch torchaudio torchvision torchcodec

## Load the Problem Words

In [ ]:
import pandas as pd

problem_words = pd.read_csv("tts_exercise/problem_words.csv", sep="\t")
problem_words.head(20)

## Build Test Sentences

In [ ]:
def build_sentence(word):
    return f"Here is the word {word}. The word {word} is often mispronounced."

TOP_N = 1000   # synthesize all problem words, capped at 1000
sentences = [(w, build_sentence(w)) for w in problem_words["word"].head(TOP_N)]

for w, s in sentences[:5]:
    print(f"{w!r}: {s}")

## TTS Model 1 : KittenTTS

In [ ]:
from kittenml import KittenTTS

kitten = KittenTTS("KittenML/kitten-tts-2-mini")
print(kitten.available_voices)

for v in kitten.voices("en"):
    print(v["name"], "-", v["description"])

In [ ]:
import soundfile as sf
from pathlib import Path
from IPython.display import Audio, display

KITTEN_DIR = Path("generated/kitten")
KITTEN_DIR.mkdir(parents=True, exist_ok=True)

KITTEN_VOICE = "Eleanor"   # <-- choose a voice from the list above

for w, sentence in sentences:
    audio = kitten.generate(sentence, voice=KITTEN_VOICE)
    out_path = KITTEN_DIR / f"{w}.wav"
    sf.write(out_path.as_posix(), audio, 24000)

# listen to the first generated sentence
display(Audio(str(KITTEN_DIR / f"{sentences[0][0]}.wav")))

## Build the Synthesized Dataset (Manifest + Train/Dev Split)

In [ ]:
import random

# one manifest row per synthesized sentence: audio path, the entity word, the transcript
dataset = []
for w, sentence in sentences:
    wav_path = KITTEN_DIR / f"{w}.wav"
    if wav_path.exists():
        dataset.append({"audio": wav_path.as_posix(), "word": w, "text": sentence})

random.seed(42)
random.shuffle(dataset)

DEV_FRACTION = 0.2   # fraction of the synthesized data held out for evaluation
n_dev = max(1, int(len(dataset) * DEV_FRACTION))
dev_split, train_split = dataset[:n_dev], dataset[n_dev:]

for name, split in [("synthesized_train.tsv", train_split), ("synthesized_dev.tsv", dev_split)]:
    with open(name, "w", encoding="utf-8") as f:
        for row in split:
            f.write(f"{row['audio']}\t{row['word']}\t{row['text']}\n")

print(f"{len(train_split)} train / {len(dev_split)} dev sentences written to synthesized_train.tsv / synthesized_dev.tsv")

## TTS Model 2 : Audio8 (Voice Cloning)

In [ ]:
import torch
import soundfile as sf
from transformers import AutoModel, AutoProcessor

model_id = "Edge0/Audio8-TTS-Realistic-Air"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" else torch.float32

processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
model = AutoModel.from_pretrained(model_id, trust_remote_code=True, dtype=dtype).eval().to(device)

In [ ]:
REFERENCE_WAV = "reference.wav"   # <-- adapt: a short reference recording
REFERENCE_TEXT = "The exact transcript of the reference recording."   # <-- its transcript

AUDIO8_DIR = Path("generated/audio8")
AUDIO8_DIR.mkdir(parents=True, exist_ok=True)

for w, sentence in sentences:
    inputs = processor(
        text=[sentence],
        reference_audio=[REFERENCE_WAV],
        reference_text=[REFERENCE_TEXT],
        return_tensors="pt",
    )
    inputs = {name: value.to(device) for name, value in inputs.items()}

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=1024,
            temperature=0.8,
            top_p=0.95,
            top_k=50,
            do_sample=True,
            return_dict_in_generate=True,
        )
        waveforms, waveform_lengths = model.decode_audio(output.codes)

    audio = waveforms[0, : int(waveform_lengths[0])].float().cpu().numpy()
    out_path = AUDIO8_DIR / f"{w}.wav"
    sf.write(out_path.as_posix(), audio, model.config.codec_sample_rate)

# listen to the first generated sentence
display(Audio(str(AUDIO8_DIR / f"{sentences[0][0]}.wav")))

## Listening Task : Are the Problem Words Spoken Correctly?

In [ ]:
# Listen to the generated sentences for the top problem words.
# For each word, judge: is the word spoken correctly and clearly?

for w, sentence in sentences:
    print("=" * 70)
    print("WORD     :", w)
    print("SENTENCE :", sentence)

    display(Audio(str(KITTEN_DIR / f"{w}.wav")))
    display(Audio(str(AUDIO8_DIR / f"{w}.wav")))